In [16]:
import random
import torch
import mypreprocess as mpp
import mylib

In [84]:
## 读取长序列数据
# 随机采样
def seq_data_iter_random(corpus, batch_size, num_steps):
    """随机抽样生成小批量子序列"""
    # 随机偏移量 范围[0, num_steps-1]
    corpus = corpus[random.randint(0, num_steps - 1):]
    # 考虑标签，确保最后一个子序列有对应标签
    num_subseqs = (len(corpus) - 1) // num_steps
    # 子序列起始索引
    initial_indices = list(range(0, num_subseqs * num_steps, num_steps))

    random.shuffle(initial_indices)

    def data(pos):
        return corpus[pos: pos + num_steps]
    
    num_batches = num_subseqs // batch_size
    for i in range(0, batch_size * num_batches, batch_size):
        initial_indices_per_batch = initial_indices[i: i + batch_size]
        X = [data(j) for j in initial_indices_per_batch]
        Y = [data(j + 1) for j in initial_indices_per_batch]
        yield torch.tensor(X), torch.tensor(Y)

# 顺序分区
def seq_data_iter_sequential(corpus, batch_size, num_steps):
    offset = random.randint(0, num_steps - 1)
    # -1 为了确保最后一个子序列有对应标签
    # 这里确保能被batch_size整除(先//后*)，是因为后面用到了
    # reshape(batch_size, -1)
    num_tokens = ((len(corpus) - offset - 1) // batch_size) * batch_size 
    Xs = torch.tensor(corpus[offset: offset + num_tokens])
    Ys = torch.tensor(corpus[offset + 1: offset + 1 + num_tokens])
    Xs, Ys = Xs.reshape(batch_size, -1), Ys.reshape(batch_size, -1)
    num_batches = Xs.shape[1] // num_steps
    for i in range(0, num_steps * num_batches, num_steps):
        X = Xs[:, i: i + num_steps]
        Y = Ys[:, i: i + num_steps]
        yield X, Y

def load_data_time_machine(batch_size, num_steps,
                           use_random_iter=False, max_tokens=10000):
    """返回时光机器数据集的迭代器和词表"""
    data_iter = SeqDataLoader(
        batch_size, num_steps, use_random_iter, max_tokens)
    return data_iter, data_iter.vocab


class SeqDataLoader:
    """加载序列数据的迭代器"""
    def __init__(self, batch_size, num_steps, use_random_iter, max_tokens):
        if use_random_iter:
            self.data_iter_fn = seq_data_iter_random
        else:
            self.data_iter_fn = seq_data_iter_sequential
        self.corpus, self.vocab = mpp.load_corpus_time_machine(max_tokens)
        self.batch_size, self.num_steps = batch_size, num_steps

    def __iter__(self):
        return self.data_iter_fn(self.corpus, self.batch_size, self.num_steps)
        


In [20]:
tokens = mpp.tokenize(mpp.read_time_machine())
corpus = [token for line in tokens for token in line]
vocab = mpp.Vocab(corpus)
vocab.token_freqs[:10]


[('the', 2261),
 ('i', 1267),
 ('and', 1245),
 ('of', 1155),
 ('a', 816),
 ('to', 695),
 ('was', 552),
 ('in', 541),
 ('that', 443),
 ('my', 440)]

In [77]:
my_seq = list(range(44))
for X, Y in seq_data_iter_random(my_seq, batch_size=2, num_steps=5):
    print('X: ', X, '\nY:', Y)

X:  tensor([[12, 13, 14, 15, 16],
        [37, 38, 39, 40, 41]]) 
Y: tensor([[13, 14, 15, 16, 17],
        [38, 39, 40, 41, 42]])
X:  tensor([[ 7,  8,  9, 10, 11],
        [27, 28, 29, 30, 31]]) 
Y: tensor([[ 8,  9, 10, 11, 12],
        [28, 29, 30, 31, 32]])
X:  tensor([[22, 23, 24, 25, 26],
        [32, 33, 34, 35, 36]]) 
Y: tensor([[23, 24, 25, 26, 27],
        [33, 34, 35, 36, 37]])
X:  tensor([[17, 18, 19, 20, 21],
        [ 2,  3,  4,  5,  6]]) 
Y: tensor([[18, 19, 20, 21, 22],
        [ 3,  4,  5,  6,  7]])


In [83]:
for X, Y in seq_data_iter_sequential(my_seq, batch_size=2, num_steps=5):
    print('X: ', X, '\nY:', Y)

X:  tensor([[ 2,  3,  4,  5,  6],
        [22, 23, 24, 25, 26]]) 
Y: tensor([[ 3,  4,  5,  6,  7],
        [23, 24, 25, 26, 27]])
X:  tensor([[ 7,  8,  9, 10, 11],
        [27, 28, 29, 30, 31]]) 
Y: tensor([[ 8,  9, 10, 11, 12],
        [28, 29, 30, 31, 32]])
X:  tensor([[12, 13, 14, 15, 16],
        [32, 33, 34, 35, 36]]) 
Y: tensor([[13, 14, 15, 16, 17],
        [33, 34, 35, 36, 37]])
X:  tensor([[17, 18, 19, 20, 21],
        [37, 38, 39, 40, 41]]) 
Y: tensor([[18, 19, 20, 21, 22],
        [38, 39, 40, 41, 42]])
